In [2]:
import pandas as pd
import spacy
import json
from tqdm import tqdm

In [2]:
geonames = pd.read_csv('../datasets/geonames.csv')

C:\Users\deepp\AppData\Local\Temp\ipykernel_41284\808585750.py:1: DtypeWarning: Columns (9,10,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  geonames = pd.read_csv('../datasets/geonames.csv')


In [21]:
jp = pd.read_csv('../datasets/added_locs.csv')

In [22]:
jp['latitude'] = None
jp['longitude'] = None

for index, row in tqdm(jp.iterrows(), total=jp.shape[0], desc="Processing locations"):
    location = row['location']  

    japan_location = geonames[geonames['country code'] == 'JP']
    location_in_japan = japan_location[japan_location['asciiname'] == location]

    if not location_in_japan.empty:
        jp.at[index, 'latitude'] = location_in_japan.iloc[0]['latitude']
        jp.at[index, 'longitude'] = location_in_japan.iloc[0]['longitude']
    else:
        other_locations = geonames[geonames['asciiname'] == location]
        if not other_locations.empty:
            jp.at[index, 'latitude'] = other_locations.iloc[0]['latitude']
            jp.at[index, 'longitude'] = other_locations.iloc[0]['longitude']

empty_locations = jp[jp['latitude'].isnull() | jp['longitude'].isnull()]
print(empty_locations)

Processing locations: 100%|██████████| 76/76 [00:51<00:00,  1.47it/s]

               location latitude longitude
2               Chuetsu     None      None
3              Chunichi     None      None
8           Hachijojima     None      None
12              Hideaki     None      None
14           Hinatazaka     None      None
17             Hokuriku     None      None
18        Honshu island     None      None
19               Honshu     None      None
20              Honshus     None      None
23  Ishikawa prefecture     None      None
31              Kishida     None      None
41             Naruhito     None      None
43   Niigata Prefecture     None      None
44             Nogizaka     None      None
47       Noto peninsula     None      None
48             Notojima     None      None
49              Onigiri     None      None
50            Onominato     None      None
52            Sakamichi     None      None
53           Sakurazaka     None      None
56           Shiga Town     None      None
57           Shika Town     None      None
62         

In [23]:
jp

,location,latitude,longitude
0,Anamizu,37.23333,136.9
1,Asahi,35.90469,136.6628
2,Chuetsu,None,None
3,Chunichi,None,None
4,Eita,46.36667,10.25
...,...,...,...
71,Yamagata,38.23333,140.36667
72,Yamamura,None,None
73,Yuka,30.79855,95.0856
74,Hashi,34.96667,132.15


In [24]:
from geopy.geocoders import Nominatim


In [25]:
geolocator = Nominatim(user_agent="location_finder")


In [26]:
for index, row in tqdm(jp.iterrows(), total=jp.shape[0], desc="Processing locations"):
    location = row['location']  

    japan_location = geonames[geonames['country code'] == 'JP']
    location_in_japan = japan_location[japan_location['asciiname'] == location]

    if not location_in_japan.empty:
        jp.at[index, 'latitude'] = location_in_japan.iloc[0]['latitude']
        jp.at[index, 'longitude'] = location_in_japan.iloc[0]['longitude']
    else:
        other_locations = geonames[geonames['asciiname'] == location]
        if not other_locations.empty:
            jp.at[index, 'latitude'] = other_locations.iloc[0]['latitude']
            jp.at[index, 'longitude'] = other_locations.iloc[0]['longitude']
        else:
            try:
                location_coords = geolocator.geocode(location)
                if location_coords:
                    jp.at[index, 'latitude'] = location_coords.latitude
                    jp.at[index, 'longitude'] = location_coords.longitude
            except Exception as e:
                print(f"Error geocoding {location}: {e}")

empty_locations = jp[jp['latitude'].isnull() | jp['longitude'].isnull()]
print(empty_locations)

Processing locations: 100%|██████████| 76/76 [01:03<00:00,  1.20it/s]

          location latitude longitude
14      Hinatazaka     None      None
47  Noto peninsula     None      None
56      Shiga Town     None      None
69     Wajima-nuri     None      None


In [30]:
jp

,location,latitude,longitude
0,Anamizu,37.23333,136.9
1,Asahi,35.90469,136.6628
2,Chuetsu,35.645037,139.801408
3,Chunichi,35.178894,136.897119
4,Eita,46.36667,10.25
...,...,...,...
71,Yamagata,38.23333,140.36667
72,Yamamura,35.696134,139.667096
73,Yuka,30.79855,95.0856
74,Hashi,34.96667,132.15


In [31]:
empty_locations = jp[jp['latitude'].isnull() | jp['longitude'].isnull()]
for index, row in empty_locations.iterrows():
    location = row['location']
    
    print(f"\nLocation: {location}")
    user_action = input("Do you want to (d)iscard this location or (m)anually enter coordinates? (d/m): ").strip().lower()

    if user_action == 'd':
        jp = jp.drop(index)
        print(f"Location {location} has been discarded.")
    elif user_action == 'm':
        try:
            latitude = float(input(f"Enter the latitude for {location}: "))
            longitude = float(input(f"Enter the longitude for {location}: "))
            jp.at[index, 'latitude'] = latitude
            jp.at[index, 'longitude'] = longitude
            print(f"Coordinates for {location} have been updated.")
        except ValueError:
            print("Invalid input. Skipping coordinates entry for this location.")
    else:
        print("Invalid option. Skipping this location.")


Location: Shiga Town
Coordinates for Shiga Town have been updated.


In [34]:
LAT_MIN, LAT_MAX = 24.396308, 45.551483
LON_MIN, LON_MAX = 122.93457, 153.986672

def is_in_japan(latitude, longitude):
    if LAT_MIN <= latitude <= LAT_MAX and LON_MIN <= longitude <= LON_MAX:
        return True
    return False

out_of_japan = []

for index, row in jp.iterrows():
    latitude = row['latitude']
    longitude = row['longitude']

    if pd.notnull(latitude) and pd.notnull(longitude):
        if not is_in_japan(latitude, longitude):
            out_of_japan.append(row)

if out_of_japan:
    out_of_japan_df = pd.DataFrame(out_of_japan)
    print("\nLocations outside of Japan:")
    print(out_of_japan_df[['location', 'latitude', 'longitude']])
else:
    print("All locations are within Japan.")


Locations outside of Japan:
      location   latitude   longitude
4         Eita  46.366670   10.250000
7     Giappone  43.773410   11.252120
12     Hideaki -21.142899  -44.224474
16        Hoje  -9.166670   15.633330
20     Honshus  60.004187   18.214535
27       Kaiso  31.600000   92.800000
29       Kanto  62.500000   22.700000
30      Kimana  -8.350000   26.950000
41    Naruhito -23.298209  -45.936769
46        Noto -14.540610   12.611490
49     Onigiri  52.231665   21.015639
57  Shika Town  22.957700  109.529000
62      Suzuki  41.562178    9.284467
73        Yuka  30.798550   95.085600


In [35]:
for index, row in out_of_japan_df.iterrows():
    location = row['location']
    print(f"\nLocation: {location}, Latitude: {row['latitude']}, Longitude: {row['longitude']}")
    action = input("Do you want to (d)iscard this location, (m)odify coordinates, or (s)kip? (d/m/s): ").strip().lower()
    
    if action == 'd':
        jp = jp.drop(index)
        print(f"Location {location} discarded.")
    elif action == 'm':
        try:
            new_latitude = float(input(f"Enter new latitude for {location}: "))
            new_longitude = float(input(f"Enter new longitude for {location}: "))
            jp.at[index, 'latitude'] = new_latitude
            jp.at[index, 'longitude'] = new_longitude
            print(f"Updated coordinates for {location}.")
        except ValueError:
            print("Invalid input. Skipping modification.")
    elif action == 's':
        print(f"Skipped {location}.")
    else:
        print("Invalid option. Skipping.")




Location: Eita, Latitude: 46.36667, Longitude: 10.25
Location Eita discarded.

Location: Giappone, Latitude: 43.77341, Longitude: 11.25212
Updated coordinates for Giappone.

Location: Hideaki, Latitude: -21.14289889748653, Longitude: -44.224473793528176
Location Hideaki discarded.

Location: Hoje, Latitude: -9.16667, Longitude: 15.63333
Location Hoje discarded.

Location: Honshus, Latitude: 60.0041867, Longitude: 18.2145354
Updated coordinates for Honshus.

Location: Kaiso, Latitude: 31.6, Longitude: 92.8
Updated coordinates for Kaiso.

Location: Kanto, Latitude: 62.5, Longitude: 22.7
Updated coordinates for Kanto.

Location: Kimana, Latitude: -8.35, Longitude: 26.95
Location Kimana discarded.

Location: Naruhito, Latitude: -23.298209149999998, Longitude: -45.9367687
Location Naruhito discarded.

Location: Noto, Latitude: -14.54061, Longitude: 12.61149
Updated coordinates for Noto.

Location: Onigiri, Latitude: 52.2316649, Longitude: 21.0156387
Location Onigiri discarded.

Location: S

In [36]:
jp

,location,latitude,longitude
0,Anamizu,37.23333,136.9
1,Asahi,35.90469,136.6628
2,Chuetsu,35.645037,139.801408
3,Chunichi,35.178894,136.897119
5,Fuji,35.16667,138.68333
...,...,...,...
70,Yabai,35.2677,138.91656
71,Yamagata,38.23333,140.36667
72,Yamamura,35.696134,139.667096
74,Hashi,34.96667,132.15


In [37]:
import pandas as pd
import folium


# Create a base map centered at a default location (e.g., Tokyo, Japan)
m = folium.Map(location=[35.6762, 139.6503], zoom_start=5)

# Iterate through the rows in jp to add markers to the map
for index, row in jp.iterrows():
    latitude = row['latitude']
    longitude = row['longitude']
    
    if pd.notna(latitude) and pd.notna(longitude):  # Only plot valid locations
        folium.Marker([latitude, longitude], popup=row['location']).add_to(m)

# Save the map to an HTML file
m.save('../maps/jp_locations_map.html')

print("Map has been saved as 'jp_locations_map.html'")


Map has been saved as 'jp_locations_map.html'


In [38]:
city = pd.read_csv('../datasets/city.csv')
countries = pd.read_csv('../datasets/countries.csv')

In [39]:
city.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 371 entries, 0 to 370
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         371 non-null    int64  
 1   geonameid          371 non-null    int64  
 2   name               371 non-null    object 
 3   asciiname          371 non-null    object 
 4   alternatenames     360 non-null    object 
 5   latitude           371 non-null    float64
 6   longitude          371 non-null    float64
 7   feature class      371 non-null    object 
 8   feature code       371 non-null    object 
 9   country code       371 non-null    object 
 10  cc2                15 non-null     object 
 11  admin1 code        371 non-null    object 
 12  admin2 code        42 non-null     object 
 13  admin3 code        1 non-null      float64
 14  admin4 code        0 non-null      float64
 15  population         371 non-null    int64  
 16  elevation          15 non-

In [40]:
countries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 36 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Unnamed: 0                                 111 non-null    int64  
 1   Country                                    111 non-null    object 
 2   Density(P/Km2)                             111 non-null    object 
 3   Abbreviation                               111 non-null    object 
 4   Agricultural Land( %)                      110 non-null    object 
 5   Land Area(Km2)                             111 non-null    object 
 6   Armed Forces size                          111 non-null    object 
 7   Birth Rate                                 111 non-null    float64
 8   Calling Code                               111 non-null    float64
 9   Capital/Major City                         110 non-null    object 
 10  Co2-Emissions             

In [41]:
city.rename(columns={'asciiname': 'location'}, inplace=True)

combined_df = pd.concat([jp, city], ignore_index=True)

combined_df = combined_df[['location', 'latitude', 'longitude']]


In [42]:
combined_df

,location,latitude,longitude
0,Anamizu,37.23333,136.9
1,Asahi,35.90469,136.6628
2,Chuetsu,35.645037,139.801408
3,Chunichi,35.178894,136.897119
4,Fuji,35.16667,138.68333
...,...,...,...
432,Eastern Cape,-32.0,26.0
433,Gauteng,-26.08333,28.25
434,Limpopo,-24.0,29.5
435,Republic of Zambia,-14.33333,28.5


In [43]:
combined_df = combined_df.drop_duplicates()

In [44]:
combined_df

,location,latitude,longitude
0,Anamizu,37.23333,136.9
1,Asahi,35.90469,136.6628
2,Chuetsu,35.645037,139.801408
3,Chunichi,35.178894,136.897119
4,Fuji,35.16667,138.68333
...,...,...,...
432,Eastern Cape,-32.0,26.0
433,Gauteng,-26.08333,28.25
434,Limpopo,-24.0,29.5
435,Republic of Zambia,-14.33333,28.5


In [45]:
countries.rename(columns={'asciiname': 'location'}, inplace=True)

coordinates = pd.concat([combined_df, countries], ignore_index=True)

coordinates = coordinates[['location', 'latitude', 'longitude']]

In [46]:
coordinates

,location,latitude,longitude
0,Anamizu,37.23333,136.9
1,Asahi,35.90469,136.6628
2,Chuetsu,35.645037,139.801408
3,Chunichi,35.178894,136.897119
4,Fuji,35.16667,138.68333
...,...,...,...
541,NaN,NaN,NaN
542,NaN,NaN,NaN
543,NaN,NaN,NaN
544,NaN,NaN,NaN


In [47]:
coordinates = coordinates.drop_duplicates()

In [48]:
coordinates

,location,latitude,longitude
0,Anamizu,37.23333,136.9
1,Asahi,35.90469,136.6628
2,Chuetsu,35.645037,139.801408
3,Chunichi,35.178894,136.897119
4,Fuji,35.16667,138.68333
...,...,...,...
431,Gauteng,-26.08333,28.25
432,Limpopo,-24.0,29.5
433,Republic of Zambia,-14.33333,28.5
434,Republic of Zimbabwe,-19.0,29.75


In [49]:
coordinates = coordinates.dropna()

In [50]:
coordinates

,location,latitude,longitude
0,Anamizu,37.23333,136.9
1,Asahi,35.90469,136.6628
2,Chuetsu,35.645037,139.801408
3,Chunichi,35.178894,136.897119
4,Fuji,35.16667,138.68333
...,...,...,...
430,Eastern Cape,-32.0,26.0
431,Gauteng,-26.08333,28.25
432,Limpopo,-24.0,29.5
433,Republic of Zambia,-14.33333,28.5


In [ ]:
coordinates.to_csv('../datasets/coordinates.csv')

In [4]:
coordinates = pd.read_csv('../datasets/coordinates.csv')

In [3]:
nlp = spacy.load("model_1.2j.2")

C:\Users\deepp\AppData\Roaming\Python\Python312\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
C:\Users\deepp\AppData\Roaming\Python\Python312\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [5]:
import json
import pandas as pd
from collections import Counter
from tqdm import tqdm
import folium

input_jsonl = "../datasets/test/5xjp.jsonl" 
with open(input_jsonl, 'r') as f:
    data = [json.loads(line) for line in f]

locations = []
for entry in data:
    text = entry[0]
    doc = nlp(text) 
    for ent in doc.ents:
        if ent.label_ == "GPE": 
            locations.append(ent.text.strip())

locations_normalized = [loc.lower() for loc in locations]
locations_count = Counter(locations_normalized)

reference_csv = "../datasets/coordinates.csv"  
reference_df = pd.read_csv(reference_csv)

geocoded_data = []
for location, count in tqdm(locations_count.items(), desc="Matching locations"):
    match = reference_df[reference_df['location'].str.lower() == location]
    if not match.empty:
        geocoded_data.append({
            "location": location,
            "latitude": match.iloc[0]['latitude'],
            "longitude": match.iloc[0]['longitude'],
            "frequency": count
        })

geocoded_df = pd.DataFrame(geocoded_data)

geocoded_df = geocoded_df.drop_duplicates()

Matching locations: 100%|██████████| 231/231 [00:00<00:00, 2224.13it/s]


In [25]:
if not geocoded_df.empty:
    avg_lat = geocoded_df['latitude'].mean()
    avg_lon = geocoded_df['longitude'].mean()
    location_map = folium.Map(location=[avg_lat, avg_lon], zoom_start=5)

    for _, row in geocoded_df.iterrows():
        marker_size = row['frequency']  
        color_intensity = min(int(row['frequency'] * 10), 255)  
        marker_color = f'#{255:02x}{255 - color_intensity:02x}{0:02x}'

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']], 
            radius=max(5, marker_size/20)+5, 
            popup=f"{row['location']} - Frequency: {row['frequency']}",
            color=marker_color,  
            fill=True,
            fill_color=marker_color,
            fill_opacity=0.7
        ).add_to(location_map)

    output_html = "../maps/map.html"  
    location_map.save(output_html)
    print(f"Map saved to {output_html}")
else:
    print("No valid locations to plot.")


Map saved to ../maps/map.html
